In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, f1_score

2025-09-12 04:57:16.315793: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [68]:
data_dir = "../data/cloud_dataset"

In [69]:
img_size = (128, 128)
batch_size = 32
seed = 123

In [70]:
# Load dataset with 80% train + 20% temp (val+test)
full_train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size,
    label_mode="int"
)

Found 5304 files belonging to 2 classes.
Using 4244 files for training.


In [71]:
temp_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=seed,
    image_size=img_size,
    batch_size=batch_size,
    label_mode="int"
)


Found 5304 files belonging to 2 classes.
Using 1060 files for validation.


In [72]:
# Split temp_ds (20%) into validation (10%) and test (10%)
val_size = 0.5
val_ds = temp_ds.take(int(len(temp_ds) * val_size))
test_ds = temp_ds.skip(int(len(temp_ds) * val_size))

In [73]:
normalization_layer = layers.Rescaling(1./255)
full_train_ds = full_train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

In [74]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation="relu", input_shape=img_size + (3,)),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.5),  # more dropout before final layer
    layers.Dense(1, activation="sigmoid")
])

/home/ilesh-dhall/cloud-burst-prediction/venv/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [75]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [76]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

In [77]:
history = model.fit(
    full_train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=[early_stop]
)

Epoch 1/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 17s 77ms/step - accuracy: 0.8230 - loss: 0.4775 - val_accuracy: 0.8603 - val_loss: 0.3846
Epoch 2/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 7s 52ms/step - accuracy: 0.8883 - loss: 0.3203 - val_accuracy: 0.8952 - val_loss: 0.2887
Epoch 3/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - accuracy: 0.9039 - loss: 0.2815 - val_accuracy: 0.9099 - val_loss: 0.3080
Epoch 4/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 4s 32ms/step - accuracy: 0.9156 - loss: 0.2588 - val_accuracy: 0.8676 - val_loss: 0.3065
Epoch 5/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - accuracy: 0.9149 - loss: 0.2439 - val_accuracy: 0.9173 - val_loss: 0.2439
Epoch 6/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 4s 32ms/step - accuracy: 0.9178 - loss: 0.2266 - val_accuracy: 0.9191 - val_loss: 0.2397
Epoch 7/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - accuracy: 0.9232 - loss: 0.2349 - val_accuracy: 0.8952 - val_loss: 0.2899
Epoch 8/100
133/133 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - accuracy: 0.9187 - loss: 0.2316 -

In [78]:
loss, acc = model.evaluate(test_ds)
print(f"Test Accuracy: {acc:.4f}")

17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.9399 - loss: 0.1536
Test Accuracy: 0.9399


In [79]:
# Collect all test images and labels in one pass
x_test, y_true = [], []
for x, y in test_ds:
    x_test.append(x.numpy())
    y_true.append(y.numpy())

x_test = np.concatenate(x_test, axis=0)
y_true = np.concatenate(y_true, axis=0)

# Predict
y_pred_probs = model.predict(x_test)
y_pred = (y_pred_probs > 0.5).astype("int32").flatten()

# F1 score
f1 = f1_score(y_true, y_pred)
print(f"Test F1-score: {f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=4))


17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 253ms/step
Test F1-score: 0.9445

Classification Report:
              precision    recall  f1-score   support

           0     0.8821    0.9712    0.9245       208
           1     0.9791    0.9123    0.9445       308

    accuracy                         0.9360       516
   macro avg     0.9306    0.9417    0.9345       516
weighted avg     0.9400    0.9360    0.9365       516



In [80]:
model.save("../models/cnn_binary_classification_model.keras")

In [17]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image

# Load model
model = tf.keras.models.load_model("../models/cnn_binary_classification_model.keras")

# Load and preprocess image
img_path = "test/NCB2.jpg"
img_size = (128, 128)

img = image.load_img(img_path, target_size=img_size)
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)
img_array = img_array / 255.0

# Predict probability
prob = model.predict(img_array)[0][0]

# Convert probability → class (0 or 1)
pred_class = int(prob > 0.5)

print(f"Predicted probability: {prob:.4f}")
print(f"Predicted class: {pred_class}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 327ms/step
Predicted probability: 0.1904
Predicted class: 0
